# 04: test one evaluation batch with the en arm
base, en_s0, en_s1, en_s2 answer all 1,200 prompts; both Qwen3Guard judges label them. Writes answers and labels only.

In [1]:
# Shared helper functions (AGENT_PROMPT_EN 0.8) -- identical in every notebook that uses them.
import re as _re

ANSWER_MAX_NEW_TOKENS = 1024
ANSWER_MAX_MODEL_LEN = 4096
JUDGE_MAX_NEW_TOKENS = 128


def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some transformers versions, a dict in others."""
    if isinstance(x, dict) or hasattr(x, "keys"):
        x = x["input_ids"]
    return [int(t) for t in x]


def load_llm(model_id, dtype, gpu_memory_utilization, max_model_len=ANSWER_MAX_MODEL_LEN, enable_lora=False, seed=0, **kw):
    from vllm import LLM
    extra = dict(enable_lora=True, max_lora_rank=16, max_loras=3) if enable_lora else {}
    return LLM(model=model_id, dtype=dtype, gpu_memory_utilization=gpu_memory_utilization,
               max_model_len=max_model_len, seed=seed, **extra, **kw)


def answer(model, prompts, lora_request=None, max_new_tokens=ANSWER_MAX_NEW_TOKENS, temperature=0.0, top_p=1.0, seed=None):
    """vLLM, chat template with a single user message, greedy by default. Returns text, token count, finish_reason."""
    from vllm import SamplingParams
    from vllm.inputs import TokensPrompt
    tok = model.get_tokenizer()
    reqs = [TokensPrompt(prompt_token_ids=_ids(tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False)))
        for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, seed=seed)
    outs = model.generate(reqs, sp, lora_request=lora_request, use_tqdm=False)
    return [{"answer": o.outputs[0].text, "tokens": len(o.outputs[0].token_ids),
             "finish_reason": o.outputs[0].finish_reason} for o in outs]


_SAFETY = _re.compile(r"^Safety:\s*(Safe|Unsafe|Controversial)\s*$", _re.MULTILINE)
_CATS = _re.compile(r"^Categories:\s*(.*)$", _re.MULTILINE)
_REFUSAL = _re.compile(r"^Refusal:\s*(Yes|No)\s*$", _re.MULTILINE)


def parse_judge(raw):
    s, c, r = _SAFETY.search(raw), _CATS.search(raw), _REFUSAL.search(raw)
    label = s.group(1) if s else None
    return {"label": label if label else "Unparsed", "categories": c.group(1).strip() if c else None,
            "refused": (1 if r.group(1) == "Yes" else 0) if r else None,
            "unsafe": 1 if label == "Unsafe" else (0 if label == "Safe" else None), "raw": raw}


def judge(judge_model, prompt, answer):
    """Qwen3Guard response moderation. prompt/answer may be a str or equal-length lists (batched)."""
    from vllm import SamplingParams
    single = isinstance(prompt, str)
    P, A = ([prompt], [answer]) if single else (list(prompt), list(answer))
    tok = judge_model.get_tokenizer()
    texts = [tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                     tokenize=False, add_generation_prompt=True) for p, a in zip(P, A)]
    outs = judge_model.generate(texts, SamplingParams(temperature=0.0, max_tokens=JUDGE_MAX_NEW_TOKENS), use_tqdm=False)
    res = [parse_judge(o.outputs[0].text) for o in outs]
    return res[0] if single else res


_JA_CHARS = _re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LATIN = _re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(text):
    """'ja' if (kana + kanji) / (Latin + kana + kanji) > 0.3, else 'en'."""
    j, l = len(_JA_CHARS.findall(text or "")), len(_LATIN.findall(text or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


In [2]:
ARM = "en"
WORK = "/content/pt3"
BATCH = 1


In [3]:
# Parameters
ARM = "en"
BATCH = 1


In [4]:
# Environment: packages, GPU, dtype, versions. HF_TOKEN comes only from the environment and is never printed.
import importlib.util, subprocess, sys, os, json, hashlib, time, random, gc
from pathlib import Path
need = [p for p, m in [("vllm", "vllm"), ("peft", "peft")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import torch, numpy as np, pandas as pd, transformers, vllm, peft
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set in the kernel environment"
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
HARDWARE = "G4" if ("RTX PRO 6000" in GPU_NAME or "Blackwell" in GPU_NAME) else ("T4" if "T4" in GPU_NAME else GPU_NAME)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
VERSIONS = {"torch": torch.__version__, "vllm": vllm.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "python": sys.version.split()[0]}
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | hardware {HARDWARE} | dtype {DTYPE}")
print(VERSIONS)
RANDOM_SEED = 0
TINY_AYA = "CohereLabs/tiny-aya-water"
GUARD_06 = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_4B = "Qwen/Qwen3Guard-Gen-4B"
QWEN_WRITER = "Qwen/Qwen3-4B-Instruct-2507"
SIM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def model_sha(mid):
    from huggingface_hub import HfApi
    return HfApi().model_info(mid).sha


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def append_jsonl(p, rows):
    with open(p, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (102 GB) | hardware G4 | dtype bfloat16
{'torch': '2.14.0+cu130', 'vllm': '0.33.0', 'transformers': '5.20.2', 'peft': '0.21.2', 'python': '3.13.16'}


In [5]:
# Colab ships torchao 0.10, which PEFT >= 0.21 rejects with an ImportError while dispatching LoRA layers.
# torchao is not used here (no quantization), so remove it before peft is used.
import importlib.metadata as _md
try:
    _tao = _md.version("torchao")
except _md.PackageNotFoundError:
    _tao = None
if _tao is not None:
    print("uninstalling torchao", _tao)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)


In [6]:
# Test one batch with this arm: base + en_s0/en_s1/en_s2 answers, judged by both guards. No rates are computed here (4.6).
BNAME = f"b{BATCH:04d}"
BDIR = Path(WORK) / "eval_pool" / "batches" / BNAME
ADAPTERS = Path(WORK) / f"arm_{ARM}" / "adapters"
EVAL = Path(WORK) / f"arm_{ARM}" / "eval"
STAGE = Path(WORK) / f"arm_{ARM}" / "eval_work" / BNAME
EVAL.mkdir(parents=True, exist_ok=True)
STAGE.mkdir(parents=True, exist_ok=True)

# Validate the batch (DONE = sha256(manifest.json); every file hash in the manifest).
assert (BDIR / "DONE").exists(), "batch has no DONE"
assert (BDIR / "DONE").read_text().strip() == sha256_file(BDIR / "manifest.json"), "DONE does not match manifest.json"
bman = json.loads((BDIR / "manifest.json").read_text())
for f, meta in bman["files"].items():
    assert sha256_file(BDIR / f) == meta["sha256"], f"hash mismatch: {f}"
prompts = load_jsonl(BDIR / "prompts.jsonl")
NP = len(prompts)
assert NP == bman["counts"]["prompts"] and NP > 0
amanifest = json.loads((ADAPTERS / "manifest.json").read_text())
DECISIONS = sorted([json.loads(p.read_text()) for p in (Path(WORK) / "eval_pool" / "decisions").glob("*.json")], key=lambda d: str(d.get("id")))
DECISION_IDS = [d["id"] for d in DECISIONS if int(d.get("applies_from_batch", 1)) <= BATCH]
SERVING = "merged" if "D002" in DECISION_IDS else amanifest["serving"]
print("decisions applied:", DECISION_IDS)
MODELS = ["base", f"{ARM}_s0", f"{ARM}_s1", f"{ARM}_s2"]
reuse_base = bman["hardware"] == HARDWARE and bman["dtype"] == DTYPE
print(f"{BNAME} (owner {bman['owner']}): {len(prompts)} prompts; serving {SERVING}; reuse baseline: {reuse_base}")

ANS = STAGE / "answers.jsonl"
ans = {(r["model"], r["prompt_id"]): r for r in load_jsonl(ANS)}
if reuse_base:
    for r in load_jsonl(BDIR / "baseline.jsonl"):
        ans.setdefault(("base", r["prompt_id"]), {"model": "base", "prompt_id": r["prompt_id"], "answer": r["answer"], "tokens": r["tokens"],
                                                  "finish_reason": r["finish_reason"], "base_source": "batch_baseline"})


def free_llm(llm):
    eng = getattr(llm, "llm_engine", None)
    core = getattr(eng, "engine_core", None)
    if core is not None and hasattr(core, "shutdown"):
        core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()


def run_answers(llm, model_name, lora=None):
    todo = [p for p in prompts if (model_name, p["prompt_id"]) not in ans]
    for i in range(0, len(todo), 400):
        chunk = todo[i:i + 400]
        outs = answer(llm, [p["prompt"] for p in chunk], lora_request=lora)
        rows = [{"model": model_name, "prompt_id": p["prompt_id"], **o, "base_source": "this_session" if model_name == "base" else None}
                for p, o in zip(chunk, outs)]
        append_jsonl(ANS, rows)
        ans.update({(r["model"], r["prompt_id"]): r for r in rows})
        print(f"  {model_name}: {i + len(chunk)}/{len(todo)}")


need_models = [m for m in MODELS if any((m, p["prompt_id"]) not in ans for p in prompts)]
print("models still to answer:", need_models)
if need_models:
    if SERVING == "vllm_lora":
        from vllm.lora.request import LoRARequest
        llm = load_llm(TINY_AYA, DTYPE, gpu_memory_utilization=0.40, enable_lora=True)
        for m in need_models:
            if m == "base":
                run_answers(llm, m)
            else:
                s = int(m[-1])
                run_answers(llm, m, LoRARequest(m, s + 1, str(ADAPTERS / f"seed{s}")))
        free_llm(llm)
    else:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import PeftModel
        TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
        for m in need_models:
            if m == "base":
                path = TINY_AYA
            else:
                s = int(m[-1])
                path = Path(WORK) / "merged" / f"seed{s}"
                if not (path / "config.json").exists():
                    b = AutoModelForCausalLM.from_pretrained(TINY_AYA, dtype=TORCH_DTYPE, device_map="cpu")
                    PeftModel.from_pretrained(b, str(ADAPTERS / f"seed{s}")).merge_and_unload().save_pretrained(path)
                    AutoTokenizer.from_pretrained(TINY_AYA).save_pretrained(path)
                    del b
                    gc.collect()
                path = str(path)
            llm = load_llm(path, DTYPE, gpu_memory_utilization=0.40)
            run_answers(llm, m)
            free_llm(llm)

assert all((m, p["prompt_id"]) in ans for m in MODELS for p in prompts)

# Judge every answer (including base) with both guards, one guard loaded at a time.
by_pid = {p["prompt_id"]: p for p in prompts}
keys = [(m, p["prompt_id"]) for m in MODELS for p in prompts]
verdicts = {}
for tag, gid, util in [("g06", GUARD_06, 0.15), ("g4b", GUARD_4B, 0.30)]:
    path = STAGE / f"judge_{tag}.jsonl"
    v = {(r["model"], r["prompt_id"]): r for r in load_jsonl(path)}
    todo = [k for k in keys if k not in v]
    print(f"{tag}: {len(v)} judged, {len(todo)} to go")
    if todo:
        g = load_llm(gid, DTYPE, gpu_memory_utilization=util, max_model_len=8192)
        for i in range(0, len(todo), 800):
            chunk = todo[i:i + 800]
            js = judge(g, [by_pid[k[1]]["prompt"] for k in chunk], [ans[k]["answer"] for k in chunk])
            rows = [{"model": k[0], "prompt_id": k[1], **j} for k, j in zip(chunk, js)]
            append_jsonl(path, rows)
            v.update({(r["model"], r["prompt_id"]): r for r in rows})
            print(f"  {tag}: {i + len(chunk)}/{len(todo)}")
        free_llm(g)
    verdicts[tag] = v

META = ["seed_id", "batch", "category", "subtopic", "form", "severity", "kind", "expected", "culture", "framing", "lang"]
out_rows = []
for m, pid in keys:
    a, p = ans[(m, pid)], by_pid[pid]
    row = {"model": m, "prompt_id": pid, **{k: p[k] for k in META}, "answer": a["answer"], "tokens": a["tokens"],
           "finish_reason": a["finish_reason"], "answer_lang": answer_lang(a["answer"]),
           "base_source": a.get("base_source") if m == "base" else None}
    for tag in ("g06", "g4b"):
        j = verdicts[tag][(m, pid)]
        for f in ("label", "categories", "unsafe", "refused", "raw"):
            row[f"{tag}_{f}"] = j[f]
    out_rows.append(row)

# Checks: 1,200 rows per model and no missing labels.
cnt = pd.Series([r["model"] for r in out_rows]).value_counts().to_dict()
assert all(cnt.get(m) == NP for m in MODELS), cnt
assert all(r["g06_label"] is not None and r["g4b_label"] is not None for r in out_rows), "missing labels"
OUTF = EVAL / f"{BNAME}.jsonl"
with open(OUTF, "w", encoding="utf-8") as f:
    for r in out_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
emani = {"batch": BATCH, "arm": ARM, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "hardware": HARDWARE,
         "dtype": DTYPE, "versions": VERSIONS, "serving": SERVING, "decisions_applied": DECISION_IDS, "base_reused_from_batch": reuse_base,
         "batch_manifest_sha256": sha256_file(BDIR / "manifest.json"),
         "models": {"base": {"id": TINY_AYA, "sha": model_sha(TINY_AYA)}, "g06": {"id": GUARD_06, "sha": model_sha(GUARD_06)},
                    "g4b": {"id": GUARD_4B, "sha": model_sha(GUARD_4B)}},
         "adapters": {m: amanifest["seeds"][str(int(m[-1]))]["files"] for m in MODELS[1:]} if "seeds" in amanifest else None,
         "rows": len(out_rows), "rows_per_model": cnt, "n_unparsed_g06": sum(r["g06_label"] == "Unparsed" for r in out_rows),
         "n_unparsed_g4b": sum(r["g4b_label"] == "Unparsed" for r in out_rows), "sha256": sha256_file(OUTF)}
(EVAL / f"{BNAME}.manifest.json").write_text(json.dumps(emani, indent=1))
print(json.dumps({k: emani[k] for k in ("rows", "rows_per_model", "n_unparsed_g06", "n_unparsed_g4b", "base_reused_from_batch")}, indent=1))
assert OUTF.stat().st_size > 0


W1114 10:23:58.967000 8347 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead


W1114 10:23:58.967000 8347 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead


W1114 10:23:58.968000 8347 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


decisions applied: ['D001', 'D002']
b0001 (owner en): 1200 prompts; serving merged; reuse baseline: True
models still to answer: ['en_s0', 'en_s1', 'en_s2']


(MainProcess pid=8347) INFO 11-14 10:24:00 [startup.py:96] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'model': '/content/pt3/merged/seed0'}


(MainProcess pid=8347) WARNING 11-14 10:24:00 [utils.py:150] Cohere2Config contains a deprecated attribute name 'rotary_pct'. Please use the preferred attribute name 'partial_rotary_factor' instead.


... (60 outputs left out)

  g4b: 4000/4800


  g4b: 4800/4800
(MainProcess pid=8347) INFO 11-14 10:32:00 [core_client.py:792] [shutdown] MPClient: start timeout=default


(MainProcess pid=8347) INFO 11-14 10:32:00 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore


{
 "rows": 4800,
 "rows_per_model": {
  "base": 1200,
  "en_s0": 1200,
  "en_s1": 1200,
  "en_s2": 1200
 },
 "n_unparsed_g06": 0,
 "n_unparsed_g4b": 0,
 "base_reused_from_batch": true
}
